In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import os

# ==========================================
# 1. Load the Protein Language Model
# ==========================================
print("Loading Model...")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D")

# Move model to GPU if available (speeds things up massively!)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

def get_protein_embedding(sequence):
    """Converts a string of amino acids into a numerical vector."""
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()} # Send inputs to GPU
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Move the result back to the CPU so numpy and pandas can read it
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

# ==========================================
# 2. Load the REAL Kaggle FASTA Data
# ==========================================
def load_fasta(file_path, limit=None):
    """Reads a .fasta file and returns a dictionary of {ID: Sequence}."""
    proteins = {}
    count = 0
    print(f"Opening file: {file_path}")
    
    with open(file_path, 'r') as f:
        protein_id = ""
        sequence = ""
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if protein_id:
                    proteins[protein_id] = sequence
                    count += 1
                    if limit and count >= limit:
                        return proteins
                # Extract just the ID (e.g., ">P12345" becomes "P12345")
                protein_id = line[1:].split()[0] 
                sequence = ""
            else:
                sequence += line
        # Catch the very last protein in the file
        if protein_id and (not limit or count < limit):
            proteins[protein_id] = sequence
            
    return proteins

import os

# 1. Force Python to hunt down the exact file path automatically
fasta_file_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'testsuperset.fasta':
            fasta_file_path = os.path.join(dirname, filename)
            break

if not fasta_file_path:
    raise FileNotFoundError("Could not find the file! Try clicking 'Remove' on the CAFA 6 dataset in the right panel, then click 'Add Data' to reattach it.")

print(f"\n✅ Automatically found the true file path: {fasta_file_path}")

# 2. Now run the loader using the dynamically found path
print("Loading test dataset...")
test_proteins = load_fasta(fasta_file_path, limit=100)
print(f"Successfully loaded {len(test_proteins)} proteins.")
# Load ONLY the first 100 proteins for a quick test run
print("\nLoading test dataset...")
test_proteins = load_fasta(fasta_file_path, limit=100)
print(f"Successfully loaded {len(test_proteins)} proteins.")

# ==========================================
# 3. Simulate Predictions & Format Submission
# ==========================================
# We will use the exact same logic as before to create the submission file.
target_go_terms = ["GO:0005575", "GO:0008150", "GO:0003674"] 
results = []

print("\nProcessing Proteins and generating vectors...")
for protein_id, sequence in test_proteins.items():
    # Generate the vector
    vector = get_protein_embedding(sequence)
    
    # Simulate a confidence score
    simulated_scores = np.random.rand(len(target_go_terms)) 
    
    for go_term, score in zip(target_go_terms, simulated_scores):
        results.append({
            "Protein_ID": protein_id,
            "GO_Term": go_term,
            "Confidence": round(score, 3) 
        })

# Save the final file
submission_df = pd.DataFrame(results)
filename = "submission.tsv"
submission_df.to_csv(filename, sep='\t', index=False, header=False)

print(f"\nSuccess! Saved to {filename}.")

**Extracting the 2D Structural Graph from the ESM-2 model.****

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# ==========================================
# 1. Load Model with "output_attentions=True"
# ==========================================
# This is the secret switch. It forces the model to expose its internal graph logic.
print("Loading ESM-2 Model...")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D", output_attentions=True)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}\n")

# ==========================================
# 2. The Extractor Function
# ==========================================
def get_2d_attention_graph(sequence):
    """Extracts the averaged 2D attention matrix for a given protein sequence."""
    
    # Tokenize the input
    inputs = tokenizer(sequence, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # outputs.attentions contains the graph data for every single layer of the neural network.
    # We stack them all together into one massive tensor.
    all_layer_attentions = torch.stack(outputs.attentions) 
    
    # We take the mathematical mean (average) across all network layers and attention heads 
    # to create one single, unified 2D relationship graph.
    mean_attention = all_layer_attentions.mean(dim=(0, 2)).squeeze(0)
    
    # CRITICAL DEBUGGING STEP: 
    # The tokenizer adds a <cls> token at the start and an <eos> token at the end.
    # We must slice the matrix [1:-1, 1:-1] to remove them so the graph exactly matches the amino acids.
    final_2d_graph = mean_attention[1:-1, 1:-1] 
    
    return final_2d_graph.cpu().numpy()

# ==========================================
# 3. Test and Debug
# ==========================================
# Let's test it on a short, 18-amino-acid protein
test_sequence = "MKALIVLGLVLLSVTVQG"
sequence_length = len(test_sequence)

print(f"Testing Sequence: {test_sequence}")
print(f"Expected Graph Size: {sequence_length} x {sequence_length}")

# Run the function
graph_matrix = get_2d_attention_graph(test_sequence)

print(f"Actual Graph Size:   {graph_matrix.shape[0]} x {graph_matrix.shape[1]}")

# Quick sanity check
if graph_matrix.shape[0] == sequence_length:
    print("\n✅ SUCCESS: The 2D Graph matches the protein length perfectly!")
else:
    print("\n❌ ERROR: Dimension mismatch.")

**Packaging the 1D and 2D Data (The PyTorch Dataset)**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. The PyTorch Data Factory
# ==========================================
class ProteinGraphDataset(Dataset):
    def __init__(self, sequences_dict):
        """
        Takes a dictionary of {Protein_ID: Sequence}
        """
        self.protein_ids = list(sequences_dict.keys())
        self.sequences = list(sequences_dict.values())
        
    def __len__(self):
        return len(self.protein_ids)
        
    def __getitem__(self, idx):
        prot_id = self.protein_ids[idx]
        seq = self.sequences[idx]
        
        # Tokenize and move to GPU
        inputs = tokenizer(seq, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Run the model to get both hidden states (1D) and attentions (2D)
        with torch.no_grad():
            outputs = model(**inputs)
            
        # --- EXTRACT 1D FEATURE VECTOR ---
        # Average the sequence to get one single vector representation
        embedding_1d = outputs.last_hidden_state.mean(dim=1).squeeze(0)
        
        # --- EXTRACT 2D GRAPH MATRIX ---
        # Average the attentions and slice off the <cls> and <eos> tokens
        all_layer_attentions = torch.stack(outputs.attentions)
        graph_2d = all_layer_attentions.mean(dim=(0, 2)).squeeze(0)[1:-1, 1:-1]
        
        # We return the tensors to the CPU here so the DataLoader can batch them efficiently
        return {
            "protein_id": prot_id,
            "feature_1d": embedding_1d.cpu(),
            "graph_2d": graph_2d.cpu()
        }

# ==========================================
# 2. Test the Factory
# ==========================================
# Let's test it with two dummy proteins
dummy_data = {
    "P12345": "MKALIVLGL",
    "Q67890": "MTEITAAMV"
}

print("Initializing Dataset...")
my_dataset = ProteinGraphDataset(dummy_data)

# Let's pull the first protein out of our factory
sample = my_dataset[0]

print(f"\nProtein ID: {sample['protein_id']}")
print(f"1D Vector Shape: {sample['feature_1d'].shape} (Ready for standard Neural Network)")
print(f"2D Graph Shape:  {sample['graph_2d'].shape} (Ready for Graph Neural Network)")

**Script 3: The Gene Ontology DAG Parser**

In [ ]:
!pip install obonet networkx

import obonet
import networkx as nx
import os

# ==========================================
# 1. Find the OBO file dynamically 
# ==========================================
# We use the same bulletproof search trick as before
obo_file_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'go-basic.obo':
            obo_file_path = os.path.join(dirname, filename)
            break

if not obo_file_path:
    raise FileNotFoundError("Could not find go-basic.obo! Make sure CAFA 6 data is attached.")

print(f"\n✅ Found Gene Ontology file at: {obo_file_path}")
print("Parsing the DAG... (This might take a few seconds)")

# ==========================================
# 2. Parse into a NetworkX Graph
# ==========================================
# This one line reads the entire biological rulebook
go_graph = obonet.read_obo(obo_file_path)

# ==========================================
# 3. Analyze the Graph
# ==========================================
print("\n--- Gene Ontology Graph Stats ---")
print(f"Total GO Terms (Nodes): {len(go_graph)}")
print(f"Total Relationships (Edges): {go_graph.number_of_edges()}")

# ==========================================
# 4. Test the Parent-Child Relationship
# ==========================================
# Let's pick a specific GO term to see how it connects to its parents
# GO:0000016 is "lactase activity"
sample_term = 'GO:0000016' 

print(f"\n--- Testing Biological Rules for {sample_term} ---")
print(f"Term Name: {go_graph.nodes[sample_term].get('name')}")
print(f"Namespace (Branch): {go_graph.nodes[sample_term].get('namespace')}")

# In obonet, edges point from Child -> Parent (using .successors)
parents = list(go_graph.successors(sample_term)) 
print(f"\nThis term is a direct child of {len(parents)} parent term(s):")

for parent in parents:
    parent_name = go_graph.nodes[parent].get('name')
    print(f" -> {parent} ({parent_name})")

brain final by pytorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. Define the Dual-Branch Architecture
# ==========================================
class ProteinFusionNetwork(nn.Module):
    def __init__(self, num_go_terms, hidden_dim=256):
        super(ProteinFusionNetwork, self).__init__()
        
        # --- BRANCH A: 1D Vector Processor ---
        # ESM-2 8M outputs a 320-dimensional vector
        self.branch_1d = nn.Sequential(
            nn.Linear(320, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # --- BRANCH B: 2D Graph Processor ---
        # We use a 2D Convolution to look for "patterns" in the attention matrix
        self.conv2d = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        # Adaptive pooling forces ANY size protein graph down to a fixed 4x4 grid
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4)) 
        
        self.branch_2d = nn.Sequential(
            nn.Linear(32 * 4 * 4, hidden_dim), # 32 channels * 4x4 grid
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # --- FUSION: Combining both brains ---
        # 256 (from 1D) + 256 (from 2D) = 512
        self.fusion_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_go_terms) # Outputs a score for every GO term!
        )

    def forward(self, vec_1d, graph_2d):
        """Passes the data through the network."""
        
        # 1. Process 1D Vector
        out_1d = self.branch_1d(vec_1d)
        
        # 2. Process 2D Graph
        # PyTorch CNNs expect shapes like (Batch, Channels, Height, Width)
        # We add a dummy "Channel" dimension to our graph
        graph_2d = graph_2d.unsqueeze(1) 
        out_2d = F.relu(self.conv2d(graph_2d))
        out_2d = self.adaptive_pool(out_2d)
        
        # Flatten the 2D output so it can be combined with the 1D output
        out_2d = out_2d.view(out_2d.size(0), -1) 
        out_2d = self.branch_2d(out_2d)
        
        # 3. Fuse and Classify
        # Glue the two thought processes together
        fused_features = torch.cat((out_1d, out_2d), dim=1)
        final_predictions = self.fusion_classifier(fused_features)
        
        return final_predictions

# ==========================================
# 2. Test the Brain with Dummy Data
# ==========================================
# Let's pretend we are predicting 500 different GO terms
dummy_num_targets = 500 
model = ProteinFusionNetwork(num_go_terms=dummy_num_targets)

# Create fake data mimicking our Data Factory output (Batch Size of 2)
# Protein 1 is length 18, Protein 2 is length 45
dummy_1d_batch = torch.rand(2, 320) 

# Because proteins are different lengths in a batch, we pad them (simulated here)
# Let's say the longest protein in this batch is 45
dummy_2d_batch = torch.rand(2, 45, 45) 

print("Passing undefined proteins into the Dual-Branch Network...\n")

# Run the model
predictions = model(dummy_1d_batch, dummy_2d_batch)

print(f"Output Shape: {predictions.shape}")
print("Meaning: (2 Proteins, 500 GO Term Confidence Scores)")
print("\n✅ SUCCESS: The Fusion Network successfully processed both 1D and 2D features!")

Script 5: The Information Accretion (IA) Loss Function

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

# ==========================================
# 1. Define the Custom Loss Function
# ==========================================
class WeightedIALoss(nn.Module):
    def __init__(self, ia_weights_tensor):
        """
        ia_weights_tensor: A 1D PyTorch tensor containing the IA weight for each GO term, 
                           in the exact same order as the model's output layer.
        """
        super(WeightedIALoss, self).__init__()
        # Move weights to the correct device (GPU/CPU) automatically
        self.register_buffer('ia_weights', ia_weights_tensor)

    def forward(self, predictions, targets):
        """
        predictions: The raw, un-normalized outputs from our Fusion Network (Logits)
        targets: The ground truth binary labels (1 for yes, 0 for no/unknown)
        """
        # BCEWithLogitsLoss is mathematically safer and faster than using Sigmoid + BCELoss
        # The 'pos_weight' parameter is specifically designed for Extreme Multi-Label imbalance!
        
        # Calculate standard loss, but multiply the error of each GO term by its IA weight
        loss = F.binary_cross_entropy_with_logits(
            predictions, 
            targets, 
            weight=self.ia_weights, 
            reduction='mean' 
        )
        return loss

# ==========================================
# 2. Simulate the IA.tsv Loading
# ==========================================
print("Simulating IA.tsv weights for our 500 dummy GO terms...")

# In the real script, we will load pd.read_csv('/kaggle/input/.../IA.tsv')
# For this test, we generate 500 random weights. 
# Rare terms get high weights (e.g., 10.0), common terms get low weights (e.g., 0.1).
dummy_num_targets = 500
simulated_ia_weights = torch.FloatTensor(np.random.uniform(0.1, 10.0, size=dummy_num_targets))

# Initialize our custom loss engine
criterion = WeightedIALoss(simulated_ia_weights)

# ==========================================
# 3. Test the Loss Calculation
# ==========================================
# We need fake predictions (from our brain) and fake targets (ground truth)
# Batch size of 2, predicting 500 terms
fake_predictions = torch.randn(2, dummy_num_targets) 

# Fake ground truth: mostly 0s (unknown), with a few 1s (confirmed functions)
fake_targets = torch.randint(0, 2, (2, dummy_num_targets)).float()

print("Calculating Custom IA Loss...")

# Calculate the error!
loss_value = criterion(fake_predictions, fake_targets)

print(f"\nFinal Loss Value: {loss_value.item():.4f}")
print("✅ SUCCESS: The Loss Engine successfully punished the model using the IA weights!")

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np

# ==========================================
# 1. Setup Data and Labels
# ==========================================
# 4 fake proteins for our test run
dummy_training_data = {
    "P12345": "MKALIVLGL",
    "Q67890": "MTEITAAMV",
    "R11223": "ALIVLGLMK",
    "S44556": "TEITAAMVM"
}

# 4 fake sets of answers (ground truth) for our 500 GO terms
dummy_targets = torch.randint(0, 2, (4, 500)).float()

print("Initializing Data Factory...")
dataset = ProteinGraphDataset(dummy_training_data)
# Batch size of 1 is easiest for testing different sized 2D graphs
dataloader = DataLoader(dataset, batch_size=1, shuffle=True) 

# ==========================================
# 2. Setup Model, Loss Engine, and Optimizer
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the Brain
fusion_model = ProteinFusionNetwork(num_go_terms=500).to(device)

# Initialize the Scoring Engine
simulated_weights = torch.FloatTensor(np.random.uniform(0.1, 10.0, size=500)).to(device)
criterion = WeightedIALoss(simulated_weights)

# Initialize the Optimizer (The mechanic that fixes the weights)
# Learning rate (lr) dictates how fast it learns. 0.001 is standard.
optimizer = optim.Adam(fusion_model.parameters(), lr=0.001)

# ==========================================
# 3. The Actual Training Loop
# ==========================================
epochs = 5
print(f"\nStarting Training Loop for {epochs} Epochs on {device}...")

for epoch in range(epochs):
    fusion_model.train() # Tell the model it is in learning mode
    total_epoch_loss = 0
    
    for idx, batch in enumerate(dataloader):
        # 1. Get the data and move it to the GPU
        vec_1d = batch['feature_1d'].to(device)
        graph_2d = batch['graph_2d'].to(device)
        
        # Grab the correct answer key for this specific protein
        actual_answers = dummy_targets[idx:idx+1].to(device)
        
        # 2. CLEAR THE MEMORY (Mandatory in PyTorch!)
        optimizer.zero_grad()
        
        # 3. FORWARD PASS (The model guesses)
        predictions = fusion_model(vec_1d, graph_2d)
        
        # 4. CALCULATE ERROR (The loss function grades the test)
        loss = criterion(predictions, actual_answers)
        
        # 5. BACKWARD PASS (The magic of deep learning - calculate how to fix the error)
        loss.backward()
        
        # 6. OPTIMIZE (Update the weights based on the backward pass)
        optimizer.step()
        
        total_epoch_loss += loss.item()
        
    # Calculate the average loss for this entire epoch
    avg_loss = total_epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

print("\n✅ SUCCESS: Pipeline Complete! The model is successfully learning.")

Script 7: The Real Data Target Mapper

In [ ]:
import pandas as pd
import numpy as np
import os
import torch

print("Hunting for real CAFA 6 label files...")

# ==========================================
# 1. Dynamically Find the Files
# ==========================================
terms_path = None
ia_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train_terms.tsv':
            terms_path = os.path.join(dirname, filename)
        if filename == 'IA.tsv':
            ia_path = os.path.join(dirname, filename)

if not terms_path or not ia_path:
    raise FileNotFoundError("Could not find train_terms.tsv or IA.tsv. Check Kaggle input folder!")

print(f"✅ Found Labels: {terms_path}")
print(f"✅ Found Weights: {ia_path}\n")

# ==========================================
# 2. Load and Process the Targets
# ==========================================
print("Loading massive training targets into Pandas... (This may take a minute)")
train_terms = pd.read_csv(terms_path, sep='\t')
print(f"Total historical annotations found: {len(train_terms):,}")

# To prevent GPU memory crashes, we won't predict all 31,000+ terms initially.
# Let's take the top 1,500 most frequently occurring GO terms to build our model's output layer.
NUM_TARGETS = 1500

# Find the top 1500 most common GO terms
top_terms = train_terms['term'].value_counts().index[:NUM_TARGETS].tolist()

# Create a dictionary to instantly look up the index of a GO term (0 to 1499)
term_to_index = {term: idx for idx, term in enumerate(top_terms)}

print(f"Successfully isolated the top {NUM_TARGETS} targets for the neural network.")

# ==========================================
# 3. Align the Information Accretion (IA) Weights
# ==========================================
print("Aligning IA.tsv weights with our targets...")
# IA.tsv has no headers, so we name the columns manually
ia_weights_df = pd.read_csv(ia_path, sep='\t', header=None, names=['term', 'weight'])

# Create a numpy array to hold the weights in the exact same order as our top_terms
real_ia_weights = np.zeros(NUM_TARGETS)

for i, term in enumerate(top_terms):
    # Find the weight for this specific term
    weight_row = ia_weights_df[ia_weights_df['term'] == term]
    if not weight_row.empty:
        real_ia_weights[i] = weight_row['weight'].values[0]
    else:
        # If a term is missing from IA.tsv, give it a baseline weight of 1.0
        real_ia_weights[i] = 1.0 

# Convert to a PyTorch Tensor so it is ready for our Loss Engine
real_ia_weights_tensor = torch.FloatTensor(real_ia_weights)

print(f"✅ Real IA Weights Extracted! Tensor Shape: {real_ia_weights_tensor.shape}")
print(f"Sample weight for {top_terms[0]}: {real_ia_weights_tensor[0]:.4f}")

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

print("Locating real FASTA sequences...")
fasta_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train_sequences.fasta':
            fasta_path = os.path.join(dirname, filename)

if not fasta_path:
    raise FileNotFoundError("Could not find train_sequences.fasta")

print(f"✅ Found FASTA: {fasta_path}")

# ==========================================
# 1. Parse Real FASTA (Subset for testing)
# ==========================================
print("Extracting a test batch of 1,000 real proteins...")
real_sequences = {}
with open(fasta_path, 'r') as f:
    current_id = ""
    current_seq = ""
    for line in f:
        if line.startswith(">"):
            if current_id:
                real_sequences[current_id] = current_seq
                if len(real_sequences) == 1000: # Stop at 1000 for rapid pipeline testing
                    break
            # Extract UniProt ID from header (e.g., >sp|P12345|...)
            parts = line.strip().split('|')
            current_id = parts[1] if len(parts) > 1 else line.strip()[1:]
            current_seq = ""
        else:
            current_seq += line.strip()

print(f"Loaded {len(real_sequences)} real sequences.")

# ==========================================
# 2. Build the Ground Truth Matrix
# ==========================================
# Protect against Kaggle changing column names by using index positions
prot_id_col = train_terms.columns[0]
term_col = train_terms.columns[1]

def get_labels_for_protein(prot_id):
    """Creates a 1500-length array of 1s and 0s for a given protein."""
    labels = np.zeros(NUM_TARGETS)
    
    # Filter the massive train_terms dataframe for this specific protein
    prot_annotations = train_terms[train_terms[prot_id_col] == prot_id][term_col].values
    
    for term in prot_annotations:
        if term in term_to_index:
            labels[term_to_index[term]] = 1.0
    return labels

# ==========================================
# 3. The Real PyTorch Dataset (PATCHED)
# ==========================================
class CAFA6Dataset(Dataset):
    def __init__(self, sequences_dict):
        self.protein_ids = list(sequences_dict.keys())
        self.sequences = list(sequences_dict.values())
        
    def __len__(self):
        return len(self.protein_ids)
        
    def __getitem__(self, idx):
        prot_id = self.protein_ids[idx]
        seq = self.sequences[idx]
        
        # --- 1. Get the 1500 Targets ---
        targets = get_labels_for_protein(prot_id)
        target_tensor = torch.FloatTensor(targets)
        
        # --- 2. Get the 1D and 2D features from ESM-2 ---
        # FIXED: padding='max_length' forces every protein to be exactly 512 tokens long
        inputs = tokenizer(seq, return_tensors="pt", padding='max_length', truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = esm2_model(**inputs)
            
        embedding_1d = outputs.last_hidden_state.mean(dim=1).squeeze(0)
        
        all_layer_attentions = torch.stack(outputs.attentions)
        graph_2d = all_layer_attentions.mean(dim=(0, 2)).squeeze(0)[1:-1, 1:-1]
        
        return {
            "protein_id": prot_id,
            "feature_1d": embedding_1d.cpu(),
            "graph_2d": graph_2d.cpu(),
            "targets": target_tensor
        }

print("Initializing Real Data Factory...")
real_dataset = CAFA6Dataset(real_sequences)
# Batch size of 2 for testing
real_dataloader = DataLoader(real_dataset, batch_size=2, shuffle=True) 

# Pull one batch to prove it works
sample_batch = next(iter(real_dataloader))

print(f"\n✅ SUCCESS: Real Data Factory is operational!")
print(f"Batch 1D Shape: {sample_batch['feature_1d'].shape}")
print(f"Batch 2D Shape: {sample_batch['graph_2d'].shape}")
print(f"Batch Targets:  {sample_batch['targets'].shape}")

In [ ]:
import torch
import torch.optim as optim

print("Connecting the Real Data Factory to the Fusion Brain...")

# ==========================================
# 1. Initialize the Engine
# ==========================================
# Re-initialize our Dual-Branch model to accept 1500 targets
real_fusion_model = ProteinFusionNetwork(num_go_terms=NUM_TARGETS).to(device)

# Use the real IA.tsv weights we extracted earlier!
real_criterion = WeightedIALoss(real_ia_weights_tensor.to(device))

# Setup the optimizer
optimizer = optim.Adam(real_fusion_model.parameters(), lr=0.001)

# ==========================================
# 2. The Heavy-Duty Training Loop
# ==========================================
epochs = 3 # Keeping it to 3 epochs for a fast test run
print(f"\nStarting Real Training Loop for {epochs} Epochs on {device}...")

for epoch in range(epochs):
    real_fusion_model.train() 
    total_epoch_loss = 0
    
    # real_dataloader comes from Script 8!
    for batch_idx, batch in enumerate(real_dataloader):
        
        # 1. Move real data to GPU
        vec_1d = batch['feature_1d'].to(device)
        graph_2d = batch['graph_2d'].to(device)
        actual_answers = batch['targets'].to(device)
        
        # 2. Clear memory
        optimizer.zero_grad()
        
        # 3. Model predicts the 1500 GO terms
        predictions = real_fusion_model(vec_1d, graph_2d)
        
        # 4. Grade the test using real IA weights
        loss = real_criterion(predictions, actual_answers)
        
        # 5. Backpropagation
        loss.backward()
        optimizer.step()
        
        total_epoch_loss += loss.item()
        
        # Print a quick update every 50 batches so we know it hasn't crashed
        if (batch_idx + 1) % 50 == 0:
            print(f"  -> Processed {batch_idx + 1} batches...")
            
    avg_loss = total_epoch_loss / len(real_dataloader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

print("\n✅ SUCCESS: The model has learned from REAL biological data!")

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import os

print("Starting Final Inference Phase...")

# ==========================================
# 1. Locate the Test Sequences
# ==========================================
fasta_file_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'testsuperset.fasta':
            fasta_file_path = os.path.join(dirname, filename)
            break

if not fasta_file_path:
    raise FileNotFoundError("Could not find testsuperset.fasta!")

print(f"✅ Found Test Data: {fasta_file_path}")

# ==========================================
# 2. Load the Test Data
# ==========================================
def load_fasta(file_path, limit=None):
    proteins = {}
    count = 0
    with open(file_path, 'r') as f:
        protein_id = ""
        sequence = ""
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if protein_id:
                    proteins[protein_id] = sequence
                    count += 1
                    if limit and count >= limit:
                        return proteins
                parts = line[1:].split('|')
                protein_id = parts[1] if len(parts) > 1 else line[1:].split()[0]
                sequence = ""
            else:
                sequence += line
        if protein_id and (not limit or count < limit):
            proteins[protein_id] = sequence
    return proteins

# Let's test on the first 50 proteins from the test set
test_proteins = load_fasta(fasta_file_path, limit=50)
print(f"Successfully loaded {len(test_proteins)} test proteins.")

# ==========================================
# 3. AI Prediction Loop
# ==========================================
results = []

# We must put the model in "evaluation" mode (turns off dropout)
real_fusion_model.eval()

print("\nPredicting functions using Dual-Branch Neural Network...")

for protein_id, seq in test_proteins.items():
    
    # 1. Extract Features from ESM-2
    # Remember to use the exact same padding length we used in training!
    inputs = tokenizer(seq, return_tensors="pt", padding='max_length', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = esm2_model(**inputs)
        
    vec_1d = outputs.last_hidden_state.mean(dim=1)
    
    all_layer_attentions = torch.stack(outputs.attentions)
    graph_2d = all_layer_attentions.mean(dim=(0, 2))[0, 1:-1, 1:-1].unsqueeze(0)
    
    # 2. Feed Features to the Brain
    with torch.no_grad():
        raw_logits = real_fusion_model(vec_1d, graph_2d)
        
        # 3. Convert raw math into percentages (0.0 to 1.0) using Sigmoid
        probabilities = torch.sigmoid(raw_logits).squeeze().cpu().numpy()
    
    # 4. Map the 1500 scores back to their actual GO Term names
    for i, score in enumerate(probabilities):
        # Only save predictions where the AI is at least 10% confident
        # This keeps the submission file from becoming gigabytes in size!
        if score > 0.10: 
            results.append({
                "Protein_ID": protein_id,
                "GO_Term": top_terms[i], # top_terms comes from our target mapper script
                "Confidence": round(float(score), 3)
            })

# ==========================================
# 4. Save the Submission File
# ==========================================
submission_df = pd.DataFrame(results)
filename = "submission.tsv"
submission_df.to_csv(filename, sep='\t', index=False, header=False)

print(f"\n🎉 Success! AI Predictions saved to {filename}.")
print(f"Total predictions made: {len(submission_df)}")